# Control de calidad de manzanas: segmentación por instancia y estado (sana / deteriorada)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EMatCM/proyectoAprendizaje/blob/main/notebooks/baseline.ipynb)

**Qué hace este notebook**

Segmenta cada manzana de una foto con varias, clasifica cada una como sana o deteriorada con el **modelo final** (`modelos/finalbaseline.pt`), cuenta el total, las sanas y las deterioradas, dibuja el resultado, evalúa con tus fotos anotadas y puede abrir una interfaz Gradio.

Aquí **no se entrena nada** y **no se descarga el dataset**. El entrenamiento y los experimentos están en `notebooks/experimentos.ipynb`.

**Componentes externos:** ResNet18 con pesos de ImageNet (la arquitectura del clasificador) y Mask R-CNN con pesos de COCO (el detector, que no se entrena), ambos vía torchvision. Lo propio: el clasificador de estado entrenado, el filtrado de detecciones, las métricas y el análisis.

----------**Interfaz**--------------------------------------------------------

Está por defecto para usar una interfaz virtual temporal con Gradio, dicha opcion se puede cambiar para usar la interfaz de colab yendo a interruptores y cambiando true a false (Gradio) y de false a true (colab).

## 1. Clonar el repositorio (código, modelo y resultados)

In [1]:
import os
REPO = "/content/proyectoAprendizaje"
if not os.path.exists(REPO):
    !git clone https://github.com/EMatCM/proyectoAprendizaje.git {REPO}
%cd {REPO}
!git pull
!ls modelos

Cloning into '/content/proyectoAprendizaje'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 91 (delta 5), reused 14 (delta 1), pack-reused 69 (from 1)
Receiving objects: 100% (91/91), 42.79 MiB | 20.98 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/proyectoAprendizaje
Already up to date.
finalbaseline.pt


## 2. Importaciones

In [2]:
# Colab ya trae torch, torchvision y pillow. Fuera de Colab: pip install -r requirements.txt
import json, random, time
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

## 3. Interruptores, parámetros y rutas

In [3]:
# ---- Interruptor ----
USAR_COLAB = False      # True: subes una foto en la celda de Colab
USAR_GRADIO = True    # True: abre la interfaz web Gradio

# ---- Parámetros ----
SEMILLA = 42
TAM = 224
MEDIA, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASE_MALA = "rottenapples"   # clase positiva (la que se cuenta como deteriorada)

# ---- Rutas ----
REPO = "/content/proyectoAprendizaje"
MODELO_FINAL = Path(f"{REPO}/modelos/finalbaseline.pt")   # clasificador de estado (modelo final)

def fijar_semilla(s=SEMILLA):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fijar_semilla()
print("Dispositivo:", dev)

Dispositivo: cpu


## 4. Transformaciones y modelo

In [4]:
def tf_eval():
    return transforms.Compose([
        transforms.Resize((TAM, TAM)),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, STD),
    ])

def crear_modelo(n_clases=2, congelar=True):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if congelar:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, n_clases)
    return m

# Varias manzanas por foto

Flujo: **Mask R-CNN** (COCO) encuentra cada manzana y su máscara → se recorta cada una con fondo blanco → el **clasificador final** decide sana o deteriorada → se cuenta el total, las sanas y las deterioradas.

Límites conocidos: las manzanas muy podridas pueden no ser detectadas como `apple`, las que se tocan o se tapan pueden fusionarse o quedar con la máscara cortada, y en montones muy llenos el detector no encuentra todas. Revisa a mano los resultados con confianza baja.

### Parámetros de la segmentación

In [5]:
import os, time
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.transforms.functional import to_tensor

# ---- Parámetros de la segmentación ----
UMBRAL_DET = 0.5        # confianza para aceptar una manzana sin más condiciones
UMBRAL_RESCATE = 0.35   # entre este valor y UMBRAL_DET solo se acepta si NO se pega a otra manzana (rescata podridas dudosas)
UMBRAL_MASCARA = 0.5    # umbral para volver binaria la máscara suave del detector
UMBRAL_MALA = 0.5       # probabilidad mínima para decir "deteriorada"
CONF_BAJA = 0.6         # por debajo de esta confianza se avisa que el resultado es incierto
MARGEN = 0.08           # margen extra alrededor de cada recorte (8 %)
MAX_LADO = 1024         # las fotos más grandes se reducen a este lado máximo
MIN_AREA_FRAC = 0.002   # descarta detecciones menores al 0,2 % de la imagen
FRAC_CONTENIDA = 0.8    # una máscara 'contiene' a otra si cubre al menos este % del área de la otra
MAX_CONTENIDAS = 2      # una detección que contiene a 2 o más manzanas es un grupo, no una manzana
MAX_AREA_REL = 4.0      # con 6 o más manzanas seguras, una detección más grande que 4 veces la mediana es un grupo (None = desactivar)
IOU_DUPLICADA = 0.6     # dos máscaras con IoU >= esto son la misma manzana: se queda la de mayor puntaje
SOLAPE_RESCATE = 0.3    # una detección dudosa se descarta si más de este % de su área cae sobre otra ya aceptada
COLOR = {"sana": (40, 170, 60), "deteriorada": (220, 40, 40)}

### Cargar clasificador y detector

In [6]:
# ---- Clasificador de estado (modelo final) ----
assert MODELO_FINAL.exists(), (
    f"No existe {MODELO_FINAL}. Modelos disponibles: "
    f"{sorted(p.name for p in MODELO_FINAL.parent.glob('*.pt'))}. "
    "Sube el modelo a modelos/ en el repositorio y vuelve a ejecutar la celda de clonado.")
ck_clf = torch.load(MODELO_FINAL, map_location=dev)
clases_clf = ck_clf.get("clases", ["freshapples", "rottenapples"])
assert CLASE_MALA in clases_clf, f"Clases del modelo: {clases_clf}"
clasificador = crear_modelo(len(clases_clf), congelar=False)
clasificador.load_state_dict(ck_clf["state_dict"])
clasificador = clasificador.to(dev).eval()
print("Clasificador:", MODELO_FINAL.name, "| Clases:", clases_clf)

# ---- Detector / segmentador (externo: Mask R-CNN preentrenado en COCO, no se entrena) ----
pesos_det = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
detector = maskrcnn_resnet50_fpn_v2(weights=pesos_det).to(dev).eval()
ID_MANZANA = pesos_det.meta["categories"].index("apple")
print("Detector cargado | id de 'apple' en COCO:", ID_MANZANA)

# El detector COCO llama 'donut', 'orange' o 'cake' a muchas manzanas muy podridas (marrones y arrugadas).
# Se aceptan esas clases como "posible manzana"; 'bowl' y otras no, porque darían falsos positivos.
CLASES_FRUTA = ["apple", "orange", "donut", "cake"]
IDS_FRUTA = torch.tensor([pesos_det.meta["categories"].index(c) for c in CLASES_FRUTA], device=dev)
print("Clases aceptadas como manzana:", CLASES_FRUTA)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 98.6MB/s]


Clasificador: finalbaseline.pt | Clases: ['freshapples', 'rottenapples']
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth


100%|██████████| 177M/177M [00:01<00:00, 144MB/s]


Detector cargado | id de 'apple' en COCO: 53
Clases aceptadas como manzana: ['apple', 'orange', 'donut', 'cake']


### Segmentar, recortar y clasificar

In [7]:
def filtrar_instancias(puntajes, mascaras, umbral_det):
    """Índices (por puntaje decreciente) de las detecciones que son UNA manzana.

    Trabaja con las máscaras, no con las cajas: una caja grande puede tocar muchas manzanas sin ser un grupo.
    1) quita grupos: máscaras que contienen a 2 o más manzanas seguras, o que son enormes frente a las demás;
    2) quita duplicados: de dos máscaras casi iguales deja la de mayor puntaje;
    3) las de puntaje dudoso (entre UMBRAL_RESCATE y umbral_det) solo se aceptan si no se montan sobre otra.
    """
    n = len(puntajes)
    if n == 0:
        return np.zeros(0, dtype=int)
    M = np.asarray(mascaras).reshape(n, -1).astype(np.float32)
    area = M.sum(1)
    inter = M @ M.T                                   # píxeles en común entre cada par de máscaras
    seguras = np.asarray(puntajes) >= umbral_det
    vivos = np.ones(n, dtype=bool)
    mediana = np.median(area[seguras]) if seguras.sum() >= 6 else None
    for i in range(n):
        dentro = sum(1 for j in range(n)
                     if j != i and seguras[j] and area[j] < 0.6 * area[i] and inter[i, j] >= FRAC_CONTENIDA * area[j])
        enorme = MAX_AREA_REL is not None and mediana is not None and area[i] > MAX_AREA_REL * mediana
        if dentro >= MAX_CONTENIDAS or enorme:
            vivos[i] = False
    kept = []
    for i in np.argsort(-np.asarray(puntajes), kind="stable"):
        if not vivos[i]:
            continue
        if kept:
            k = np.array(kept)
            iou = inter[i, k] / (area[i] + area[k] - inter[i, k])
            if (iou >= IOU_DUPLICADA).any():
                continue
            if puntajes[i] < umbral_det and (inter[i, k] / area[i] > SOLAPE_RESCATE).any():
                continue
        kept.append(int(i))
    return np.array(kept, dtype=int)


@torch.no_grad()
def segmentar(img, umbral_det=None):
    """Devuelve cajas, puntajes y máscaras booleanas de cada manzana detectada."""
    umbral_det = UMBRAL_DET if umbral_det is None else umbral_det
    piso = min(UMBRAL_RESCATE, umbral_det)            # el detector entrega también las dudosas; luego se filtran
    out = detector([to_tensor(img).to(dev)])[0]
    ok = torch.isin(out["labels"], IDS_FRUTA) & (out["scores"] >= piso)
    cajas = out["boxes"][ok].cpu().numpy()
    puntajes = out["scores"][ok].cpu().numpy()
    mascaras = (out["masks"][ok, 0] > UMBRAL_MASCARA).cpu().numpy()
    area_min = MIN_AREA_FRAC * img.width * img.height
    keep = np.array([i for i, m in enumerate(mascaras) if m.sum() >= area_min], dtype=int)
    cajas, puntajes, mascaras = cajas[keep], puntajes[keep], mascaras[keep]
    keep = filtrar_instancias(puntajes, mascaras, umbral_det)
    return cajas[keep], puntajes[keep], mascaras[keep]


def recortar(arr, caja, mascara, margen=MARGEN):
    """Recorta una manzana y pone fondo blanco fuera de su máscara."""
    x1, y1, x2, y2 = caja
    mx, my = margen * (x2 - x1), margen * (y2 - y1)
    x1, y1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
    x2, y2 = min(arr.shape[1], int(x2 + mx)), min(arr.shape[0], int(y2 + my))
    sub = arr[y1:y2, x1:x2].copy()
    sub[~mascara[y1:y2, x1:x2]] = 255
    return Image.fromarray(sub)


@torch.no_grad()
def prob_deteriorada(recortes, lote=32):
    """Probabilidad de 'deteriorada' para cada recorte (el mismo clasificador del baseline)."""
    if not recortes:
        return []
    idx = clases_clf.index(CLASE_MALA)
    probs = []
    for i in range(0, len(recortes), lote):
        x = torch.stack([tf_eval()(r) for r in recortes[i:i + lote]]).to(dev)
        probs += torch.softmax(clasificador(x), dim=1)[:, idx].cpu().tolist()
    return probs

### Análisis de una imagen y resumen

In [8]:
def analizar(img, umbral_mala=None, umbral_det=None):
    """Segmenta, clasifica cada manzana y arma el resumen de la imagen."""
    umbral_mala = UMBRAL_MALA if umbral_mala is None else umbral_mala
    t0 = time.time()
    img = img.convert("RGB")
    factor = 1.0
    if max(img.size) > MAX_LADO:                      # fotos de celular muy grandes
        factor = MAX_LADO / max(img.size)
        img = img.resize((round(img.width * factor), round(img.height * factor)))

    cajas, puntajes, mascaras = segmentar(img, umbral_det)
    arr = np.asarray(img)
    recortes = [recortar(arr, c, m) for c, m in zip(cajas, mascaras)]
    probs = prob_deteriorada(recortes)

    instancias = []
    for i, (c, s, m, p) in enumerate(zip(cajas, puntajes, mascaras, probs), 1):
        mala = p >= umbral_mala
        instancias.append({
            "id": i,
            "caja": [float(v) for v in c],
            "puntaje_det": float(s),
            "prob_deteriorada": float(p),
            "estado": "deteriorada" if mala else "sana",
            "confianza": float(p if mala else 1 - p),
            "area": int(m.sum()),
            "mascara": m,
        })

    n_malas = sum(d["estado"] == "deteriorada" for d in instancias)
    avisos = []
    if not instancias:
        avisos.append("No se detectó ninguna manzana. Prueba bajar el umbral de detección.")
    dudosas = [d["id"] for d in instancias if d["confianza"] < CONF_BAJA]
    if dudosas:
        avisos.append(f"Confianza baja en las manzanas {dudosas}: revisar a mano.")
    return {"imagen": img, "factor": factor, "instancias": instancias,
            "total": len(instancias), "sanas": len(instancias) - n_malas,
            "deterioradas": n_malas, "avisos": avisos, "segundos": time.time() - t0}


def resumen_texto(res):
    lineas = [f"Manzanas: {res['total']} | sanas: {res['sanas']} | deterioradas: {res['deterioradas']} "
              f"| {res['segundos']:.1f} s"]
    for d in res["instancias"]:
        lineas.append(f"  #{d['id']}: {d['estado']} (confianza {d['confianza']:.0%}, "
                      f"detección {d['puntaje_det']:.0%})")
    lineas += [f"Aviso: {a}" for a in res["avisos"]]
    return "\n".join(lineas)

### Dibujar el resultado

In [9]:
def _borde(m, grosor=3):
    """Contorno de una máscara booleana, sin necesidad de OpenCV."""
    interior = m.copy()
    for _ in range(grosor):
        p = np.pad(interior, 1, constant_values=False)
        interior = p[1:-1, 1:-1] & p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:]
    return m & ~interior


def _fuente(tam):
    for ruta in ["/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", "DejaVuSans-Bold.ttf"]:
        try:
            return ImageFont.truetype(ruta, tam)
        except OSError:
            pass
    return ImageFont.load_default()


def dibujar(res, alpha=0.40):
    """Pinta cada manzana (verde = sana, rojo = deteriorada) con contorno y etiqueta."""
    arr = np.asarray(res["imagen"]).astype(np.float32)
    for d in res["instancias"]:
        color = COLOR[d["estado"]]
        m = d["mascara"]
        arr[m] = (1 - alpha) * arr[m] + alpha * np.array(color, dtype=np.float32)
        arr[_borde(m)] = color
    out = Image.fromarray(arr.clip(0, 255).astype(np.uint8))
    dr = ImageDraw.Draw(out)
    fuente = _fuente(max(14, out.width // 45))
    for d in res["instancias"]:
        color = COLOR[d["estado"]]
        texto = f"#{d['id']} {d['estado']} {d['confianza']:.0%}"
        x1, y1 = d["caja"][:2]
        bb = dr.textbbox((0, 0), texto, font=fuente)
        w, h = bb[2] - bb[0], bb[3] - bb[1]
        px, py = x1, max(0, y1 - h - 10)
        dr.rectangle([px, py, px + w + 10, py + h + 10], fill=color)
        dr.text((px + 5, py + 3), texto, fill="white", font=fuente)
    return out

### Probar con una foto de varias manzanas

In [10]:
if USAR_COLAB:
    from google.colab import files
    from IPython.display import display

    subida = files.upload()          # sube una foto con varias manzanas
    if not subida:
        print("No se subió ninguna imagen. Vuelve a ejecutar esta celda y elige un archivo.")
    else:
        ruta = list(subida.keys())[0]
        res = analizar(Image.open(ruta))
        print(resumen_texto(res))
        display(dibujar(res))
        os.remove(ruta)              # para que la foto no quede dentro del repositorio
else:
    print("Interfaz de Colab omitida (USAR_COLAB = False).")

Interfaz de Colab omitida (USAR_COLAB = False).


### Interfaz Gradio

In [11]:
if USAR_GRADIO:
    !pip install -q gradio
    import gradio as gr

    def interfaz(img, umbral_mala, umbral_det):
        if img is None:
            return None, "Sube una foto."
        res = analizar(img, umbral_mala, umbral_det)
        return dibujar(res), resumen_texto(res)

    demo = gr.Interface(
        fn=interfaz,
        inputs=[gr.Image(type="pil", label="Foto con varias manzanas"),
                gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Umbral para 'deteriorada'"),
                gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Umbral de detección")],
        outputs=[gr.Image(label="Resultado"), gr.Textbox(label="Resumen", lines=8)],
        title="Control de calidad de manzanas",
        description="Segmenta cada manzana y la clasifica como sana o deteriorada. Inspección solo visual.",
    )
    demo.launch(share=True)
else:
    print("Gradio omitido (USAR_GRADIO = False).")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a4eebc0d64641a97a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Dónde está el resto

- Entrenamiento del clasificador, experimentos y comparaciones: `notebooks/experimentos.ipynb`.
- Modelos y métricas guardadas: carpetas `modelos/` y `resultados/` del repositorio.